https://cursos.alura.com.br/course/series-temporais-detectando-anomalias-realizando-previsoes/task/163139

Curso prático de séries temporais aplicado a dados reais de uma rede de restaurantes. Ao longo do projeto, são exploradas análises da variação de clientes ao longo do tempo, identificação de anomalias e construção de modelos de previsão. O foco está na geração de insights para apoiar o planejamento operacional, especialmente na otimização da alocação de atendentes.

Séries temporais são conjuntos de dados organizados em ordem cronológica, ou seja, dados registrados ao longo do tempo em intervalos regulares, como dias, meses ou anos. Esses dados são utilizados para analisar como uma determinada variável se comporta ao longo do tempo.

As aplicações de séries temporais são diversas. Como exemplo, podemos citar os registros de temperatura de uma cidade ao longo dos dias, o número de produtos vendidos por uma loja a cada mês e o valor de uma ação na bolsa de valores a cada dia de negociação.


Analisar dados de séries temporais é importante para nos ajudar a entender o passado e aplicar o conhecimento adquirido para prever o que pode acontecer no futuro. Existem várias maneiras de analisar séries temporais, dentre elas:

- Visualização: Plotar os dados em um gráfico ao longo do tempo é o primeiro passo. Isso ajuda a identificar visualmente as variações dos valores durante todo o período.

- Médias Móveis: Utilizar médias móveis pode ajudar a suavizar os dados, facilitando a visualização das tendências de crescimento e decrescimento. Por exemplo, a média móvel de 7 dias de uma série diária pode mostrar a tendência semanal.

- Previsão: Treinar modelos de regressão para utilizar o padrão dos dados do passado e prever valores futuros usando machine learning.
Detecção de anomalias: Utilizar técnicas de detecção de valores discrepantes ajuda a encontrar dados distantes da média e fora do padrão geral dos dados.

- Detecção de anomalias: Utilizar técnicas de detecção de valores discrepantes ajuda a encontrar dados distantes da média e fora do padrão geral dos dados.

In [0]:
%pip install statsmodels

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import dates
import matplotlib.ticker as ticker
import plotly.express as px
from scipy.stats import zscore
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

In [0]:
###Nome dos restaurantes
### Serie temporal que varia ao longo do tempo

dados_consolidados = pd.read_csv('/Workspace/Repos/patriciatamiresdesousa@gmail.com/series_temporais_deteccao_anomalias_previsao/clientes_restaurantes.csv')
dados_consolidados

In [0]:
dados_consolidados.info()

In [0]:
##ajuste data 
dados_consolidados['data'] = pd.to_datetime(dados_consolidados['data'])

##ajuste indice
dados_consolidados['data'] = pd.to_datetime(dados_consolidados['data'])
dados_consolidados.set_index('data', inplace = True)

In [0]:
dados_consolidados

In [0]:
#### Tratamento de nulos 

dados_consolidados['Chimi & Churri'][dados_consolidados['Chimi & Churri'].isna()]

In [0]:
### Importante eliminar dados nulos em series temporais, porque alem de afetar a linha do tempo, afeta as medias e medianas nas analises 

dados_consolidados['Chimi & Churri'].plot(figsize = (20,3))
plt.axvline(x = dados_consolidados['Chimi & Churri'][dados_consolidados['Chimi & Churri'].isna()].index[0], color = 'red')
plt.axvline(x = dados_consolidados['Chimi & Churri'][dados_consolidados['Chimi & Churri'].isna()].index[1], color = 'red');

In [0]:
dados_consolidados['Assa Frão'][dados_consolidados['Assa Frão'].isna()]

Vou substituir os valores nulos por valores que correspondem ao padrão de um período. Por exemplo, posso substituir o valor nulo pelo valor do dia anterior ou pelo valor do dia posterior. Isso faz sentido porque, nesses dias próximos, a série temporal tende a apresentar um padrão semelhante.

In [0]:
### Interpolação = interpolação calcula um ponto médio entre o valor da data anterior e o valor da data posterior
###Essa função detecta automaticamente os dados nulos e realiza a substituição com base na lógica de calcular o ponto médio entre a data anterior e a data posterior. Ela preencherá apenas os valores nulos na base de dados. Ao executar essa célula, toda a base será atualizada, mas apenas os dados nulos serão preenchidos.


dados_consolidados = dados_consolidados.interpolate()

In [0]:
##ajuste deu certo nas duas datas que apresentavam nulos

dados_consolidados.loc['2016-04-04':'2016-04-06']

dados_consolidados.loc['2016-09-16':'2016-09-18']

dados_consolidados.loc['2016-11-23':'2016-11-25']

In [0]:
### quero responder algumas perguntas como média, mediana, quantidade, distribuicao dos valores 

In [0]:
##deu erro
dados_consolidados = dados_consolidados.astype(int)
dados_consolidados

In [0]:
### entendendo o erro da data
dados_consolidados.dtypes

In [0]:
###ajustado
dados_consolidados = dados_consolidados.astype(int)
dados_consolidados

In [0]:
## Distribuicao no histograma 

## quantidade mais frequente de clientes entre 20 a 40 
## quantidade menos de clientes em 100 a 120 

dados_consolidados['Chimi & Churri'].plot(kind = 'hist', title = 'Distribuição dos dados do Chimi & Churri', figsize = (10, 4));

In [0]:
dados_consolidados.describe()

In [0]:

###Observei que os valores mínimos do Chimi & Churri e do Assa Frão são bastante próximos. Em cada gráfico, a barra verde horizontal representa a mediana, ou seja, o valor intermediário da quantidade de clientes que frequentam cada restaurante.
##O ponto mais relevante da análise está nos valores acima da última linha horizontal. Os diversos círculos pretos identificados, tanto no Chimi & Churri quanto no Assa Frão, indicam a presença de valores discrepantes, que se afastam de forma significativa do padrão geral de quantidade de clientes observado em cada restaurante.


dados_consolidados.plot(kind = 'box', title = 'Boxplots da quantidade de clientes dos restaurantes');

In [0]:
### Contabilizando total de clientes de toda rede 
dados_consolidados.sum(axis = 1)

In [0]:
##Incluindo mais uma coluna no dataframe com total
dados_consolidados['Total'] = dados_consolidados.sum(axis = 1)

In [0]:
##Uma variacao de 100 a 200 clientes no total

dados_consolidados['Total'].plot(kind = 'hist', title = 'Distribuição dos dados dos restaurantes', figsize = (10,4));

In [0]:
dados_consolidados.describe

In [0]:
## Serie temporal mes, dia e ano

ax = dados_consolidados['Total'].plot(figsize = (20,3), title = 'Clientes por dia na rede de restaurantes')
ax.xaxis.set_major_locator(dates.MonthLocator())
ax.xaxis.set_major_formatter(dates.DateFormatter('%b/%Y'));

In [0]:
### Encontrando a quantidade mensal dos clientes 

clientes_mensais = dados_consolidados.resample(rule = 'M')[['Chimi & Churri', 'Assa Frão']].sum()
clientes_mensais

In [0]:
!locale -a

In [0]:
clientes_mensais.index.strftime('%b/%Y')

In [0]:
clientes_mensais['Mês'] = clientes_mensais.index.month_name()
clientes_mensais


In [0]:
### Com isso, consigo observar no gráfico de barras a variação da quantidade de clientes por mês. Inicialmente, noto que a barra vermelha, referente ao restaurante Assa Frão, apresenta uma quantidade maior de clientes em comparação ao restaurante Chimi & Churri, especialmente em janeiro de 2016.
### No entanto, ao analisar o comportamento ao longo do tempo, percebo que a quantidade de clientes do Assa Frão parece ter diminuído, já que as barras vermelhas se tornam menores. Em contrapartida, a quantidade de clientes do Chimi & Churri aparenta crescer. Essa mudança fica mais evidente a partir de março e abril de 2017.


ax = clientes_mensais.plot(x = 'Mês', kind = 'bar', stacked = True, color = ['#636EFA', '#EF553B'], rot = 0, figsize = (20,6))
ax.bar_label(ax.containers[1], fontsize = 10)
ax.xaxis.set_minor_locator(ticker.IndexLocator(12, 0))
ax.xaxis.set_minor_formatter(ticker.FixedFormatter(['\n\n2016', '\n\n2017']));

Extraindo a média móvel
Nesse caso, vou utilizar a média móvel, em que é calculado a média dos valores da quantidade de clientes em períodos de 7 dias. Essa média varia e sempre captura os 7 próximos dias.

In [0]:
for restaurante in ['Chimi & Churri', 'Assa Frão']:
    clientes_mensais[f'{restaurante}_MM7'] = (
        clientes_mensais[[restaurante]]
        .rolling(7)
        .mean()
    )

    print(clientes_mensais.columns)


#### Detecção de anomalias em séries temporais


In [0]:
### antes de encontrar as anomalias eu preciso identificar os dias da semana, se é feriado, se é fds, etc

dados_consolidados['Dia da semana'] =dados_consolidados.index.strftime('%a')

In [0]:

datas_comemorativas  = pd.read_csv('/Workspace/Repos/patriciatamiresdesousa@gmail.com/series_temporais_deteccao_anomalias_previsao/datas_comemorativas.csv')
datas_comemorativas.head()

In [0]:
###tratamento do dado data dentro do dataframe

datas_comemorativas['data'] = pd.to_datetime(datas_comemorativas['data'])
datas_comemorativas = datas_comemorativas.set_index('data', drop = True)
datas_comemorativas

In [0]:
###Merge onde eu junto o tratamento que eu fiz na tabela data com a minha tabela principal!


dados_consolidados = pd.merge(dados_consolidados, datas_comemorativas, how = 'left', left_index = True, right_index = True)
dados_consolidados

####Capturando dados discrepantes 

Dados discrepantes serão identificados com base no desvio padrão dos dados. Caso um valor se afaste muito em relação a média geral dos dados, eles são considerados um valor discrepante 

Aqui, vou utilizar o método z-score para transformar os dados e identificar o quanto cada valor está distante da média. Com essa transformação, a média dos dados passa a ser representada pelo valor zero, e os valores positivos ou negativos indicam o afastamento em relação a essa média. Dessa forma, consigo analisar o comportamento dos dados com base no seu desvio em relação ao valor central.

In [0]:
def detectar_anomalias(coluna):
    dados_consolidados['zscore'] = zscore(dados_consolidados[coluna])
    anomalias = dados_consolidados[(dados_consolidados['zscore']>3) | (dados_consolidados['zscore']<-3)]
    return anomalias[[coluna, 'zscore', 'Dia da semana', 'feriado']]

In [0]:
anomalias_chimi_churri = detectar_anomalias('Chimi & Churri')
anomalias_assa_frao = detectar_anomalias('Assa Frão')

In [0]:
### Anomalia maior que 3 e abril

ax = dados_consolidados['Chimi & Churri'].plot(label = 'Clientes', color = 'gray', figsize = (20,4))
ax.scatter(anomalias_chimi_churri.index.to_pydatetime(), anomalias_chimi_churri['Chimi & Churri'], color = 'red', label = 'Anomalia')
ax.set_title('Anomalias Chimi & Churri')
ax.legend();

In [0]:
##Identificar as datas especificas informações que podem ser uteis 

print('Anomalias para o Chimi & Churri')
display(anomalias_chimi_churri)
print('\n\n')
print('Anomalias para o Assa Frão')
display(anomalias_assa_frao)

#####Autocorrelação
Mede a relação linear entre os valores de uma série temporal e versões defasadas de si mesma, indicando como dados passados influenciam os atuais. É

In [0]:

##Autocorrelação: Atrasos das comparações vai ser levado todas as datas para frente
###Autocorrelação parcial: correlaciona apenas os 7 dias 


fig, axes = plt.subplots(1, 2, figsize = (16,5))
plot_acf(dados_consolidados['Chimi & Churri'], lags = 30, title = 'Autocorrelação', auto_ylims = True, ax = axes[0])
plot_pacf(dados_consolidados['Chimi & Churri'], lags = 30, title = 'Autocorrelação parcial', auto_ylims = True, ax = axes[1])
plt.suptitle('Autocorrelação Chimi & Churri')

#### Decompondo uma série de tempo

O que eu vou fazer é decompor a minha série temporal. Vou analisar a série e explicá-la a partir de seus principais componentes: a tendência, que representa o crescimento ou decrescimento ao longo do tempo, e a sazonalidade, que captura padrões repetitivos.

Essa abordagem vai me ajudar a entender melhor o comportamento geral da quantidade de clientes ao longo do tempo.

Para realizar essa decomposição, vou utilizar uma técnica que separa e explica cada um desses componentes da série temporal, gerando novas séries para cada parte. Isso ficará mais claro quando eu executar o código.

In [0]:
decomposicao_chimi_churri = seasonal_decompose(dados_consolidados['Chimi & Churri'])

In [0]:
###Tendencia de series temporais
decomposicao_chimi_churri.trend

In [0]:
##Sazonalidade de series temporais decomposicao_chimi_churri.seasonal
decomposicao_chimi_churri.seasonal

In [0]:
### Residos que eu não consigo explicar matematicamente, mas que estão entre meus dados
decomposicao_chimi_churri.resid